Here, cortical Glut neurons with Rbfox3 below the 10th percentile of WT (incl. Rbfox3 zeros) are defined, transferred to the full object and exported (`…-fulldata`, input for 06/07), and ECs are binarized by distance to the nearest NeuN-low neuron (fixed cutoffs 18/31/58/103 µm).

In [ ]:
import sys
from datetime import date
from pathlib import Path

import anndata as ad
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import shapely
import shapely.plotting
from matplotlib import rc_context
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy.spatial import cKDTree
from scipy.stats import ttest_ind

sys.path.insert(1, "/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark")
from cellseg_benchmark._constants import cell_type_colors
from cellseg_benchmark.adata_utils import plot_spatial_multiplot

plt.rcParams["font.family"] = "Arial" if "Arial" in [f.name for f in fm.fontManager.ttflist] else "sans-serif"
ad.settings.allow_write_nullable_strings = True

## Helper functions

In [ ]:
def plot_gene_by_genotype(a, gene, exclude_zeros=False, vline_pct=None, savepath=None):
    x = sc.get.obs_df(a, [gene, "genotype"], layer="volume_log1p_norm")
    if exclude_zeros:
        x = x[x[gene] != 0]
    lo, hi = np.percentile(x[gene], [0, 99.9])
    plt.figure(figsize=(6, 6))
    for g, color, alpha in [("ECKO", "#22799c", 1), ("WT", "#cfcfcf", 0.8)]:
        sns.histplot(x.loc[x["genotype"] == g, gene], color=color, alpha=alpha, bins=50, label=g, stat="density", linewidth=0, binrange=(lo, hi))
    if vline_pct:
        plt.axvline(np.percentile(x.loc[x["genotype"] == "WT", gene], vline_pct), linewidth=1, color="grey")
    plt.xlabel(f"{gene} (volume_log1p_norm)")
    plt.legend()
    if savepath:
        plt.savefig(savepath, bbox_inches="tight")
    plt.show()

def dotplot_values(a, genes):
    """Color: mean z-score. Size: fraction of cells with nonzero log1p expression (a z-score has no meaningful zero). Both averaged per animal."""
    z = sc.get.obs_df(a, genes + ["sample"], layer="zscore").groupby("sample", observed=True).mean()
    x = sc.get.obs_df(a, genes + ["sample"], layer="volume_log1p_norm").set_index("sample").gt(0).groupby(level=0, observed=True).mean()
    return pd.DataFrame({"mean_expression": z.mean(), "fraction_of_cells": x.mean()})

def dotplot(a, values, title, vmin, vmax, dot_max):
    dp = sc.pl.DotPlot(a, var_names=list(values.index), groupby="cohort", vmin=vmin, vmax=vmax, title=title, figsize=(2, 3))
    dp.dot_color_df = pd.DataFrame([values["mean_expression"]], index=dp.dot_color_df.index)
    dp.dot_size_df = pd.DataFrame([values["fraction_of_cells"]], index=dp.dot_size_df.index)
    dp.color_legend_title = "Mean expression\n(z-score)"
    dp.size_title = "% of cells with\nnonzero expression"
    return dp.swap_axes().style(dot_max=dot_max)

In [ ]:
base_path = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark")
method_path = base_path / "analysis/foxf2/Cellpose_1_nuclei_model"
output_path = method_path / "neuronal-subclustering"
out_dir = output_path / "dotplots"
out_dir.mkdir(parents=True, exist_ok=True)

## Load
Neuron annotation from 02.

In [ ]:
adata = sc.read_h5ad(method_path / "adatas/20251111_temp-after-neuronal-subclustering-reannotated.h5ad.gz")
cell_type_sub = adata.obs["cell_type_sub"]
adata = adata[adata.obs["brain_region"] == "CTX"].copy()
adata.obs["Rbfox3_zero"] = sc.get.obs_df(adata, ["Rbfox3"], layer="volume_log1p_norm")["Rbfox3"].eq(0)
adata_glut = adata[adata.obs["cell_type_sub"] == "Neurons-Glut"].copy()
adata_gaba = adata[adata.obs["cell_type_sub"] == "Neurons-Gaba"].copy()
pd.crosstab([adata.obs["genotype"], adata.obs["sample"]], adata.obs["slide"])

## NeuN-zero and NeuN-low neurons (cortex)

In [ ]:
plot_gene_by_genotype(adata_glut, "Rbfox3", vline_pct=10, savepath=output_path / "Rbfox3_hist_Glut.svg")
plot_gene_by_genotype(adata_gaba, "Rbfox3", exclude_zeros=True)
for by in ["genotype", ["genotype", "sample"]]:
    t = adata_glut.obs.groupby(by, observed=True)["Rbfox3_zero"].agg(["sum", "size", "mean"])
    print(f"NeuN-zero Glut neurons per {' x '.join(by) if isinstance(by, list) else by}")
    display(t.assign(pct=100 * t.pop("mean")).round(1))

In [ ]:
# NeuN-low: Rbfox3 below the 10th percentile of WT Glut neurons (incl. zeros)
expr = sc.get.obs_df(adata_glut, ["Rbfox3"], layer="volume_log1p_norm")["Rbfox3"]
thr = expr[adata_glut.obs["genotype"] == "WT"].quantile(0.1)
adata_glut.obs["Rbfox3_low"] = expr < thr
print(f"Rbfox3 low threshold: {thr:.4f}")
for by in ["genotype", ["genotype", "sample"]]:
    t = adata_glut.obs.groupby(by, observed=True)["Rbfox3_low"].agg(["sum", "size", "mean"])
    print(f"NeuN-low Glut neurons per {' x '.join(by) if isinstance(by, list) else by}")
    display(t.assign(pct=100 * t.pop("mean")).round(1))

## Dotplot Glut neurons

In [ ]:
adata_glut.layers["zscore"] = sc.pp.scale(adata_glut.layers["volume_log1p_norm"], copy=True)
genes = ["Acsl4", "Gclc", "Gclm", "Gpx4", "Slc7a11", "Slc11a2", "Tfrc", "Trf"]  # Slc11a2 = Dmt1
tables = {}
for suffix, a in {"": adata_glut, "_excl-NeuN-zero": adata_glut[~adata_glut.obs["Rbfox3_zero"]]}.items():
    ko, low = a.obs["genotype"] == "ECKO", a.obs["Rbfox3_low"]
    for name, m in {"WT_all": a.obs["genotype"] == "WT", "KO_all": ko, "KO_low": ko & low, "KO_high": ko & ~low}.items():
        tables[name + suffix] = dotplot_values(a[m], genes)
all_df = pd.concat(tables.values())
vmin, vmax = all_df["mean_expression"].min(), all_df["mean_expression"].max()
dot_max = np.ceil(all_df["fraction_of_cells"].max() * 10) / 10
for title, values in tables.items():
    dp = dotplot(adata_glut, values, title, vmin, vmax, dot_max)
    dp.make_figure()
    display(dp.fig)
    for ext in ("png", "svg"):
        dp.savefig(out_dir / f"Glut_{title}.{ext}")
    plt.close("all")
pd.concat(tables, names=["group", "gene"]).to_excel(out_dir / "Glut_dotplot_values.xlsx")

## Transfer labels to full data (WT and ECKO)

In [ ]:
adata = sc.read_h5ad(method_path / "adatas/adata_integrated.h5ad.gz")
adata = adata[adata.obs["genotype"].isin(["WT", "ECKO"])].copy()
adata.obs["cell_type"] = adata.obs["cell_type_revised"].astype(str).replace({"Astroependymal": "Astrocytes"})
adata.obs["sample2"] = adata.obs["sample"].astype(str) + "_" + adata.obs["genotype"].astype(str)
adata.obs["brain_region"] = adata.obs_names.map(pd.read_csv(method_path / "spatial_registration.csv", index_col=0)["label"])
adata.obsm["spatial"] = adata.obsm["spatial_microns"]
for c in ["Rbfox3_low", "Rbfox3_zero"]:
    adata.obs[c] = adata_glut.obs[c].reindex(adata.obs_names).astype(str).map({"True": True, "False": False})
adata.shape

In [ ]:
# NeuN-low (%), NeuN-zero (%) and NeuN+ / NeuN-low density (cells/um2, relative to Ctrl) per sample in cortex, mean +- SEM, t-test
ctx = adata.obs["brain_region"] == "CTX"
xy = adata.obsm["spatial_microns"][:, :2]
tissue = {}
for s in adata.obs.loc[ctx, "sample"].unique():
    t = shapely.MultiPoint(xy[(ctx & (adata.obs["sample"] == s)).to_numpy()]).buffer(50, quad_segs=4).buffer(-50)
    tissue[s] = shapely.union_all([shapely.Polygon(p.exterior, [h for h in p.interiors if shapely.Polygon(h).area > 5e4])
                                   for p in shapely.get_parts(t) if p.area > 5e4])
area = pd.Series({s: t.area for s, t in tissue.items()})
display(pd.DataFrame({"area_um2": area, "all_cells_per_um2": ctx.groupby(adata.obs["sample"], observed=True).sum() / area}))

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, (s, t) in zip(axes.flat, tissue.items()):
    ax.scatter(*xy[(ctx & (adata.obs["sample"] == s)).to_numpy()].T, s=0.1, c="0.6", rasterized=True)
    shapely.plotting.plot_polygon(t, ax=ax, add_points=False, facecolor="none", edgecolor="r", lw=1)
    ax.set(title=f"{s}: {t.area:.3g} µm²", aspect="equal", xticks=[], yticks=[])
for ax in axes.flat[len(tissue):]:
    ax.axis("off")
plt.show()

o = adata.obs[ctx & adata.obs["Rbfox3_low"].notna()]
low, zero, g = o["Rbfox3_low"].astype(str).eq("True"), o["Rbfox3_zero"].astype(str).eq("True"), [o["genotype"], o["sample"]]
panels = {
    "NeuN-low neurons\n(% of Glut neurons)": low.groupby(g, observed=True).mean().mul(100),
    "NeuN-zero neurons\n(% of Glut neurons)": zero.groupby(g, observed=True).mean().mul(100),
}
for k, m in {"NeuN$^+$ Glut neurons": ~low, "NeuN-low neurons": low}.items():
    d = m.groupby(g, observed=True).sum().div(area, level="sample")
    panels[f"{k}\nrel. count (cells/µm²)"] = d / d["WT"].mean()
pd.DataFrame(panels).rename(columns=lambda c: c.replace("\n", " ").replace("$^+$", "+")).join(area.rename("area_um2"), on="sample").to_csv(output_path / "lowNeuN_pct_density_per_sample.csv")
style = {"WT": ("#cfcfcf", "#4d4d4d", "o", "Ctrl"), "ECKO": ("#22799c", "#14485d", "h", "Foxf2$^{iECKO}$")}
fig, axes = plt.subplots(1, 4, figsize=(6.2, 2.4))
for ax, (ylabel, pct) in zip(axes, panels.items()):
    for x, (gt, (bar, dot, marker, label)) in enumerate(style.items()):
        v = pct[gt]
        ax.bar(x, v.mean(), yerr=v.sem(), width=0.7, color=bar, capsize=3, error_kw={"lw": 1}, label=label)
        ax.scatter(x + np.linspace(-0.15, 0.15, len(v)), v, s=20, c=dot, marker=marker, zorder=3)
    y = pct[list(style)].max() * 1.1
    ax.plot([0, 1], [y, y], c="k", lw=1)
    ax.text(0.5, y, f"p={ttest_ind(pct['ECKO'], pct['WT']).pvalue:.3g}", ha="center", va="bottom")
    ax.set_ylim(top=y * 1.15)
    ax.set_xticks([])
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, 0), ncol=2, frameon=False, handlelength=1, columnspacing=0.8)
for ext in ("png", "svg"):
    fig.savefig(output_path / f"lowNeuN_pct_density_per_sample.{ext}", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# neuronal subtypes from 02, NeuN-low Glut neurons as own cell type, colors
s = adata.obs["cell_type"].astype(object)
s.update(cell_type_sub)
counts = s.value_counts()
adata = adata[s.isin(counts[counts > 1].index)].copy()  # drop groups with 1 cell
adata.obs["cell_type"] = s
adata.obs["cell_type_incl_lowNeuN"] = adata.obs["cell_type"].where(~adata.obs["Rbfox3_low"].eq(True), "Neurons-Glut-lowNeuN")
colors_extended = {**cell_type_colors, "Neurons-Glut": "#CEB3FF", "Neurons-Gaba": "#B449F8", "Neurons-Other": "#FCA0FF",
                   "Neurons-Mixed": "#D9D9D9", "Neurons-Glut-lowNeuN": "red"}
for col, colors in {"cell_type": cell_type_colors, "cell_type_incl_lowNeuN": colors_extended}.items():
    adata.obs[col] = pd.Categorical(adata.obs[col], categories=list(colors)).remove_unused_categories()
    adata.uns[f"{col}_colors"] = [colors[c] for c in adata.obs[col].cat.categories]
adata.obs["cell_type_incl_lowNeuN"].value_counts()

In [ ]:
for c in ["Rbfox3_low", "Rbfox3_zero"]:
    adata.obs[c] = adata.obs[c].astype("category")
adata_ctx = adata[adata.obs["brain_region"] == "CTX"].copy()
for a, name in [(adata, ""), (adata_ctx, "_ctx")]:
    with rc_context({"figure.figsize": (8, 8)}):
        sc.pl.embedding(a, basis="X_umap_harmony_20_50", color=["cell_type", "cell_type_incl_lowNeuN", "Rbfox3_low", "Rbfox3_zero"],
                        size=260000 / a.shape[0], wspace=0.6, frameon=False, show=False)
        plt.savefig(output_path / f"umap_Rbfox3_low{name}.png", dpi=600, bbox_inches="tight", pad_inches=0)
        plt.show()
plot_spatial_multiplot(adata_ctx, "cell_type", size=0.2, figsize_per_ax=2)
plot_spatial_multiplot(adata_ctx[adata_ctx.obs["cell_type_incl_lowNeuN"].isin(["Neurons-Glut", "Neurons-Glut-lowNeuN"])].copy(),
                       "cell_type", size=7, figsize_per_ax=7.5, title_keys=["sample", "genotype"])

In [ ]:
adata.write(method_path / "adatas" / f"{date.today():%Y%m%d}_temp-after-neuronal-subclustering-reannotated-fulldata.h5ad.gz", compression="gzip")

## Distance of ECs to NeuN-low neurons (cortex)
ECs are binarized into proximal vs distal by the distance to the nearest NeuN-low Glut neuron (same sample), at fixed cutoffs of 18/31/58/103 µm (~10/25/50/75th percentiles). For dotplots, distances are also computed to NeuN-low neurons excl. NeuN-zero cells. DEA: see 07.

In [ ]:
adata = adata[adata.obs["brain_region"] == "CTX"].copy()
xy = adata.obsm["spatial_microns"][:, :2]
ec = (adata.obs["cell_type"] == "ECs").to_numpy()
low = (adata.obs["Rbfox3_low"].eq(True) & (adata.obs["cell_type"] == "Neurons-Glut")).to_numpy()
for suffix, src in {"": low, "_excl-NeuN-zero": low & ~adata.obs["Rbfox3_zero"].eq(True).to_numpy()}.items():
    d = pd.Series(np.nan, index=adata.obs_names)
    for s in adata.obs["sample2"].unique():
        m = (adata.obs["sample2"] == s).to_numpy()
        if (m & ec).any() and (m & src).any():
            d[m & ec] = cKDTree(xy[m & src]).query(xy[m & ec])[0]
    adata.obs[f"dist_ec_to_nearest_lowNeuN{suffix}_um"] = d
    for q, c in zip((0.1, 0.25, 0.5, 0.75), (18, 31, 58, 103)):
        adata.obs[f"ec_proximal_to_lowNeuN_{c}um{suffix}"] = d <= c
        print(f"{suffix or 'incl-NeuN-zero'} {q:.0%} quantile: {d.quantile(q):.1f} um, cutoff: {c} um")

plt.figure(figsize=(6, 4))
plt.hist(adata.obs["dist_ec_to_nearest_lowNeuN_um"].dropna(), bins=60, density=True)
plt.xlabel("EC distance to nearest lowNeuN neuron (µm)")
plt.ylabel("Density")
plt.show()

In [ ]:
# spatial plot to verify distances
sample_name, radius_um, bar_um = "foxf2_s1_r0", 18, 200
palette = {"Neurons-Glut": "#e5e5e5", "Neurons-Glut-lowNeuN": "#b2182b", "EC proximal": "#08306b", "EC distal": "#6baed6"}
a = adata[(adata.obs["sample"] == sample_name) & adata.obs["cell_type_incl_lowNeuN"].isin(["Neurons-Glut", "Neurons-Glut-lowNeuN", "ECs"])].copy()
ct = a.obs["cell_type_incl_lowNeuN"].to_numpy()
grp = np.where(ct == "Neurons-Glut-lowNeuN", "Neurons-Glut-lowNeuN", "Neurons-Glut").astype(object)
grp[ct == "ECs"] = np.where(a.obs[f"ec_proximal_to_lowNeuN_{radius_um}um"].to_numpy()[ct == "ECs"], "EC proximal", "EC distal")
a.obs["plot_group"] = pd.Categorical(grp, categories=list(palette), ordered=True)
a = a[a.obs["plot_group"].argsort()].copy()

with rc_context({"figure.figsize": (15, 15)}):
    fig = sc.pl.embedding(a, basis="spatial_microns", color="plot_group", palette=palette, size=10,
                          legend_loc="right margin", show=False, return_fig=True)
ax = fig.axes[0]
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"{sample_name}, EC radius = {radius_um} µm", fontsize=14)
ax.add_collection(PatchCollection(
    [Circle((x, y), radius_um) for x, y in a.obsm["spatial_microns"][(a.obs["cell_type_incl_lowNeuN"] == "ECs").to_numpy(), :2]],
    facecolor="none", edgecolor="black", linewidths=0.15, alpha=0.08,
))
ax.add_artist(AnchoredSizeBar(ax.transData, bar_um, f"{bar_um} µm", loc="lower right", pad=0.3, borderpad=0.5, sep=5,
                              frameon=False, size_vertical=2, fontproperties=fm.FontProperties(size=10), color="black"))
plt.tight_layout()
fig.savefig(output_path / f"{sample_name}_EC_radius_{radius_um}um_spatial.png", dpi=800, bbox_inches="tight")
plt.show()

## Dotplot ECs

In [ ]:
adata_ECs = adata[adata.obs["cell_type"] == "ECs"].copy()
adata_ECs.layers["zscore"] = sc.pp.scale(adata_ECs.layers["volume_log1p_norm"], copy=True)
gene_groups = {
    "BEC/BBB": ["Foxf2", "Tek", "Vegfa", "Vegfb", "Cldn5", "Cdh5", "Tjp1", "Tjp2"],
    "ROS/NRF2": ["Nos3", "Sod1", "Cat", "Gclm", "Gclc", "Slc7a11", "Slc3a2", "Abcg2"],
}
ko = adata_ECs.obs["genotype"] == "ECKO"
tables = {"WT_all": dotplot_values(adata_ECs[~ko], sum(gene_groups.values(), []))}
for suffix in ("", "_excl-NeuN-zero"):
    prox = adata_ECs.obs[f"ec_proximal_to_lowNeuN_18um{suffix}"].astype(bool)
    tables[f"KO_proximal_18um{suffix}"] = dotplot_values(adata_ECs[ko & prox], sum(gene_groups.values(), []))
    tables[f"KO_distal_18um{suffix}"] = dotplot_values(adata_ECs[ko & ~prox], sum(gene_groups.values(), []))

sc.settings.autoshow = False
for group_name, genes in gene_groups.items():
    # color scale symmetric around 0 (5-95% quantiles of z-scores), dot size up to the 95% quantile
    sub = pd.concat(tables.values()).loc[genes]
    m = float(sub["mean_expression"].quantile([0.05, 0.95]).abs().max())
    dot_max = round(float(sub["fraction_of_cells"].quantile(0.95)), 1)
    for title, values in tables.items():
        dp = dotplot(adata_ECs, values.loc[genes], f"{title}\n{group_name}", -m, m, dot_max)
        dp.make_figure()
        display(dp.fig)
        for ext in ("png", "svg"):
            dp.savefig(out_dir / f"{title}_{group_name.replace('/', '-')}.{ext}")
        plt.close(dp.fig)
pd.concat(tables, names=["group", "gene"]).to_excel(out_dir / "ECs_dotplot_values.xlsx")

In [ ]:
adata.write(method_path / "adatas" / f"{date.today():%Y%m%d}_temp-after_ec_proximal_to_lowNeuN.h5ad.gz", compression="gzip")